# Data Pipeline: Transcribe, Translate, Synthesize, Align

**GPU Budget:** ~6-8 hours on Kaggle T4 (per direction)

This notebook runs the complete data preparation pipeline for creating
synthetic parallel speech pairs:

```
Source Audio  -->  Whisper (timestamps + text)  -->  MADLAD translate
    -->  MMS-TTS synthesize target speech  -->  Contextual align
    -->  Silence insertion  -->  Save to Kaggle working dir
```

### Data Sources
- **Sw->En**: KenSpeech Swahili (auto-downloaded from HuggingFace)
- **En->Sw**: FLEURS en_us (auto-downloaded from HuggingFace)
- No manual dataset download required

### Run Order
Run this notebook **twice** -- once per direction:
1. First run: **Sw->En** (source=sw, target=en) -- KenSpeech source, pretrained `facebook/mms-tts-eng`
2. Second run: **En->Sw** (source=en, target=sw) -- FLEURS source, pretrained `facebook/mms-tts-swh`

**After each run:** save `/kaggle/working/hibiki-sw` output as a new Kaggle Dataset version before the session ends.

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torchaudio soundfile scipy faster-whisper sentencepiece
# Note: bitsandbytes is intentionally NOT installed. We swapped MADLAD-400 for
# NLLB-200-distilled-1.3B (~2.6 GB fp16) which fits on a single T4 without
# quantization. int8/NF4 also produced degenerate outputs on T5 models in
# past runs, so we stay fp16 end-to-end.

In [ ]:
# HuggingFace authentication (fast downloads + push_to_hub uploads)
# Add your token in Kaggle: Settings > Secrets > Add secret named "HF_TOKEN"
import os

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # legacy env var fallback

    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("HuggingFace login successful")
except Exception as e:
    print(f"HF token not set ({e}). Downloads may be rate-limited and push_to_hub will not work.")

In [ ]:
import os
import sys
import torch

REPO_DIR = '/kaggle/input/datasets/victormugambi/hibiki-sw/hibiki-sw'
sys.path.insert(0, REPO_DIR)
os.environ['PYTHONPATH'] = REPO_DIR

BASE_DIR = '/kaggle/working/hibiki-sw'
os.makedirs(BASE_DIR, exist_ok=True)

# === CHANGE THESE FOR EACH RUN ===
# Run 1: SOURCE_LANG='sw', TARGET_LANG='en'
# Run 2: SOURCE_LANG='en', TARGET_LANG='sw'
SOURCE_LANG = 'sw'
TARGET_LANG = 'en'

# Local KenSpeech dataset (attached as Kaggle dataset — no download needed)
KENSPEECH_DIR = '/kaggle/input/kenspeech-sw'

# Data source: 'kenspeech' for Sw->En, 'directory' for En->Sw (FLEURS)
SOURCE = 'kenspeech' if SOURCE_LANG == 'sw' else 'directory'

MAX_SAMPLES = 10000  # KenSpeech has ~5816, FLEURS en_us has ~2009

print(f'Direction: {SOURCE_LANG} -> {TARGET_LANG}')
print(f'Source: {SOURCE}')
print(f'KenSpeech dir: {KENSPEECH_DIR} (exists={os.path.exists(KENSPEECH_DIR)})')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Verify data source accessibility
if SOURCE == 'kenspeech':
    from data.prepare.kenspeech_loader import KenSpeechLoader
    ks = KenSpeechLoader(load_audio=False, local_dir=KENSPEECH_DIR)
    print(f'KenSpeech loaded: {len(ks)} samples')
    stats = ks.get_stats()
    print(f'Unique speakers: {stats["unique_speakers"]}')
    print(f'Avg sentence length: {stats["avg_sentence_length"]:.0f} chars')
    del ks
    print('Dataset ready!')

elif SOURCE == 'directory':
    FLEURS_AUDIO_DIR = f'{BASE_DIR}/fleurs_en_audio'
    if not os.path.exists(FLEURS_AUDIO_DIR) or len(os.listdir(FLEURS_AUDIO_DIR)) == 0:
        print('Extracting FLEURS en_us audio to WAV files...')
        os.makedirs(FLEURS_AUDIO_DIR, exist_ok=True)
        from datasets import load_dataset
        import soundfile as sf
        import numpy as np

        ds = load_dataset("google/fleurs", "en_us", split="train")
        for i, sample in enumerate(ds):
            audio = sample["audio"]
            wav_path = os.path.join(FLEURS_AUDIO_DIR, f"fleurs_en_{i:05d}.wav")
            sf.write(wav_path, np.array(audio["array"], dtype=np.float32), audio["sampling_rate"])
        print(f'Extracted {len(ds)} WAV files to {FLEURS_AUDIO_DIR}')
    else:
        n_wavs = len([f for f in os.listdir(FLEURS_AUDIO_DIR) if f.endswith('.wav')])
        print(f'FLEURS audio already extracted: {n_wavs} WAV files')
    print('Dataset ready!')

## Step 1: Whisper Transcription (~2-3 hrs for KenSpeech, ~1 hr for FLEURS)

- **Sw->En (KenSpeech)**: Uses KenSpeech's existing transcripts as text, runs Whisper only for word-level timestamps needed by the alignment step.
- **En->Sw (FLEURS)**: Runs Whisper normally for both transcription and timestamps.

In [ ]:
# Restore transcriptions from saved dataset (skip Whisper if already done)
import shutil

TRANSCRIPTS_DATASET = f'/kaggle/input/datasets/victormugambi/transcriptions'
TRANSCRIPTION_DIR = f'{BASE_DIR}/transcriptions/{SOURCE_LANG}'

existing = len([f for f in os.listdir(TRANSCRIPTION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSCRIPTION_DIR) else 0

if existing > 0:
    print(f'Transcriptions already present: {existing} files — skipping restore')
else:
    src = f'{TRANSCRIPTS_DATASET}/{SOURCE_LANG}'
    if os.path.isdir(src):
        os.makedirs(TRANSCRIPTION_DIR, exist_ok=True)
        shutil.copytree(src, TRANSCRIPTION_DIR, dirs_exist_ok=True)
        existing = len([f for f in os.listdir(TRANSCRIPTION_DIR) if f.endswith('.json')])
        print(f'Restored {existing} transcription files from dataset')
    else:
        print(f'Dataset not found at {src} — will run Whisper in the next cell')

In [ ]:
TRANSCRIPTION_DIR = f'{BASE_DIR}/transcriptions/{SOURCE_LANG}'

existing = len([f for f in os.listdir(TRANSCRIPTION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSCRIPTION_DIR) else 0
print(f'Existing transcriptions: {existing}')

# Skip Whisper if transcriptions were restored from dataset (non-zero existing files).
# resume_from is passed for partial runs, but won't skip model loading — so we only
# invoke the script when there's actual work to do.
if existing > 0 and SOURCE == 'kenspeech':
    print(f'Transcriptions already present — skipping Whisper entirely.')
elif SOURCE == 'kenspeech':
    !cd {REPO_DIR} && python data/prepare/transcribe_whisper.py \
        --source kenspeech \
        --kenspeech_dir {KENSPEECH_DIR} \
        --lang {SOURCE_LANG} \
        --output_dir {TRANSCRIPTION_DIR} \
        --whisper_model medium \
        --max_samples {MAX_SAMPLES} \
        --resume_from {existing}
else:
    !cd {REPO_DIR} && python data/prepare/transcribe_whisper.py \
        --source directory \
        --lang {SOURCE_LANG} \
        --audio_dir {FLEURS_AUDIO_DIR} \
        --output_dir {TRANSCRIPTION_DIR} \
        --whisper_model medium \
        --max_samples {MAX_SAMPLES}

In [ ]:
# Verify transcriptions
import json
from pathlib import Path

trans_files = sorted(Path(TRANSCRIPTION_DIR).glob('*.json'))
print(f'Total transcription files: {len(trans_files)}')

# Preview a sample
if trans_files:
    with open(trans_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {trans_files[0].name}')
    print(f'  Text: {sample.get("text", "")[:100]}')
    print(f'  Duration: {sample.get("audio_duration", 0):.1f}s')
    print(f'  Words: {len(sample.get("words", []))}')
    if sample.get('words'):
        print(f'  First word: {sample["words"][0]}')

## Step 2: NLLB Translation (~1-2 hrs)

Translate all transcriptions to the target language using NLLB-200-distilled-1.3B.

In [ ]:
import shutil

DIRECTION = f'{SOURCE_LANG}2{TARGET_LANG}'
TRANSLATION_DIR = f'{BASE_DIR}/translations/{DIRECTION}'
TRANSLATIONS_DATASET = '/kaggle/input/datasets/victormugambi/translations'

existing = len([f for f in os.listdir(TRANSLATION_DIR) if f.endswith('.json')]) if os.path.exists(TRANSLATION_DIR) else 0
print(f'Existing translations: {existing}')

# Restore from saved Kaggle dataset if local working dir is empty.
if existing == 0:
    src = f'{TRANSLATIONS_DATASET}/{DIRECTION}'
    if os.path.isdir(src):
        os.makedirs(TRANSLATION_DIR, exist_ok=True)
        shutil.copytree(src, TRANSLATION_DIR, dirs_exist_ok=True)
        existing = len([f for f in os.listdir(TRANSLATION_DIR) if f.endswith('.json')])
        print(f'Restored {existing} translation files from dataset')
    else:
        print(f'Saved translations not found at {src} — will run NLLB below.')

# Skip translation entirely if we already have enough files.
# (MAX_SAMPLES=10000 caps the run; "enough" = at least the cap or all available source files)
if existing >= MAX_SAMPLES or (existing > 0 and existing >= len(list(Path(TRANSCRIPTION_DIR).glob('*.json')))):
    print(f'Translations already present ({existing}) — skipping NLLB entirely.')
else:
    # Switched from MADLAD-400-3B to NLLB-200-distilled-1.3B. MADLAD produced
    # degenerate output across every dtype (int8/NF4/fp16/bf16/fp32) in this
    # Kaggle T4 environment, and the minimal HF model-card repro also failed --
    # likely a T5x->HF weight-conversion issue in the current transformers
    # version. NLLB is not T5-based, stable in fp16, ~2.6 GB weights (fits on
    # one T4), and has native swh_Latn <-> eng_Latn. Same JSON contract so
    # downstream alignment/synthesis cells are unchanged.

    # Smoke test first: 5 canonical sentences; visually confirm coherent output.
    print('\n--- Smoke test (~1 min on T4) ---')
    !cd {REPO_DIR} && python data/prepare/translate_nllb.py \
        --source_lang {SOURCE_LANG} \
        --target_lang {TARGET_LANG} \
        --dtype float16 \
        --device_map auto \
        --smoke_test

    # Full translation run (resumes from `existing`)
    print('\n--- Full translation run ---')
    !cd {REPO_DIR} && python data/prepare/translate_nllb.py \
        --input_dir {TRANSCRIPTION_DIR} \
        --output_dir {TRANSLATION_DIR} \
        --source_lang {SOURCE_LANG} \
        --target_lang {TARGET_LANG} \
        --dtype float16 \
        --device_map auto \
        --batch_size 16 \
        --max_samples {MAX_SAMPLES} \
        --resume_from {existing}

In [ ]:
# Verify translations — FAIL LOUDLY if nothing was produced
trans_files = sorted(Path(TRANSLATION_DIR).glob('*.json'))
print(f'Total translation files: {len(trans_files)}')

if len(trans_files) == 0:
    raise RuntimeError(
        f"TRANSLATION STEP PRODUCED 0 FILES in {TRANSLATION_DIR}\n"
        "Check cell 10 output for errors from translate_nllb.py.\n"
        "Possible causes:\n"
        "  - TRANSCRIPTION_DIR is empty (transcription step failed)\n"
        "  - NLLB model failed to load (OOM or HF download error)\n"
        "Do NOT continue — remaining steps have nothing to process."
    )

if trans_files:
    with open(trans_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {trans_files[0].name}')
    print(f'  Source ({SOURCE_LANG}): {sample.get("source_text", "")[:100]}')
    print(f'  Target ({TARGET_LANG}): {sample.get("translated_text", "")[:100]}')
    if not sample.get("translated_text", "").strip():
        raise RuntimeError(
            "translated_text is EMPTY in the first translation file.\n"
            "Translation ran but produced blank outputs — check NLLB logs."
        )

## Step 3: Contextual Alignment (~30 min)

Compute per-word alignment between source and target text using
NLLB-200-distilled-1.3B perplexity (Hibiki paper, Section 3.2.1, eq. 6).
The paper used MADLAD-400-3B; we substituted NLLB because MADLAD's
encoder/decoder produced broken logits in our environment. The algorithm
is model-agnostic — any encoder-decoder MT with reliable log-probs works.

This determines which source word maps to which target word, which is
needed for the silence insertion step.

In [ ]:
ALIGNMENT_DIR = f'{BASE_DIR}/alignments/{DIRECTION}'

!cd {REPO_DIR} && python data/prepare/run_pipeline.py \
    --source {SOURCE} \
    --source_lang {SOURCE_LANG} \
    --target_lang {TARGET_LANG} \
    --base_dir {BASE_DIR} \
    --max_samples {MAX_SAMPLES} \
    --step align

In [ ]:
# Verify alignments
align_files = sorted(Path(ALIGNMENT_DIR).glob('*.json'))
print(f'Total alignment files: {len(align_files)}')

if align_files:
    with open(align_files[0]) as f:
        sample = json.load(f)
    print(f'\nSample: {align_files[0].name}')
    print(f'  Source: {sample.get("source_text", "")[:80]}')
    print(f'  Target: {sample.get("translated_text", "")[:80]}')
    print(f'  Alignment pairs: {len(sample.get("alignment", []))}')
    if sample.get('alignment'):
        print(f'  First 5 pairs: {sample["alignment"][:5]}')

## Step 4: TTS Synthesis + Silence Insertion (~2-3 hrs)

This is the key step that creates the synthetic parallel speech:
1. Synthesize target-language speech from translations using MMS-TTS
2. Get word timestamps on synthesized audio with Whisper
3. Apply silence insertion for causal alignment (Hibiki paper Section 3.2)

**For Sw->En:** Uses pretrained `facebook/mms-tts-eng`
**For En->Sw:** Uses pretrained `facebook/mms-tts-swh`

In [ ]:
SYNTH_DIR = f'{BASE_DIR}/synthetic_audio/{DIRECTION}'

cmd = (
    f"cd {REPO_DIR} && python data/prepare/synthesize_tts.py"
    f" --translation_dir {TRANSLATION_DIR}"
    f" --output_dir {SYNTH_DIR}"
    f" --target_lang {TARGET_LANG}"
    f" --alignment_dir {ALIGNMENT_DIR}"
    f" --whisper_model medium"
    f" --target_sr 24000"
    f" --max_samples {MAX_SAMPLES}"
)

# No VITS model -- use pretrained MMS-TTS for both directions

print(f'Running synthesis...\n{cmd}\n')
!{cmd}

In [ ]:
# Verify synthesized audio — FAIL LOUDLY if synthesis produced nothing
import IPython.display as ipd
import torchaudio

aligned_dir = f'{SYNTH_DIR}/aligned_audio'
raw_dir = f'{SYNTH_DIR}/raw_synthesis/wavs'

# Check which directory has output
if os.path.exists(aligned_dir) and len(list(Path(aligned_dir).glob('*.wav'))) > 0:
    wav_dir = aligned_dir
else:
    wav_dir = raw_dir

n_wavs = len(list(Path(wav_dir).glob('*.wav'))) if os.path.exists(wav_dir) else 0
print(f'Synthesized WAVs in {wav_dir}: {n_wavs} total')

if n_wavs == 0:
    # Print diagnostic info to help debug
    trans_files = list(Path(TRANSLATION_DIR).glob('*.json')) if os.path.exists(TRANSLATION_DIR) else []
    print(f'\nDiagnostic:')
    print(f'  Translation files: {len(trans_files)} in {TRANSLATION_DIR}')
    print(f'  SYNTH_DIR exists: {os.path.exists(SYNTH_DIR)}')
    raw_synthesis_dir = f'{SYNTH_DIR}/raw_synthesis'
    print(f'  raw_synthesis dir exists: {os.path.exists(raw_synthesis_dir)}')
    if os.path.exists(raw_synthesis_dir):
        print(f'  raw_synthesis contents: {os.listdir(raw_synthesis_dir)}')
    raise RuntimeError(
        f"SYNTHESIS PRODUCED 0 AUDIO FILES in {wav_dir}\n"
        "Check cell 15 output for errors from synthesize_tts.py.\n"
        "Possible causes:\n"
        "  - Translation dir was empty (0 JSON files to synthesize)\n"
        "  - MMS-TTS model failed to load (OOM or HF download error)\n"
        "  - All synthesis calls failed silently (check 'Failed: N' count in cell 15)\n"
        "Do NOT continue to encoding — there is nothing to encode."
    )

wav_files = sorted(Path(wav_dir).glob('*.wav'))[:5]
print()
for wav_path in wav_files:
    waveform, sr = torchaudio.load(str(wav_path))
    dur = waveform.shape[1] / sr
    print(f'{wav_path.name} ({dur:.2f}s, {sr}Hz)')
    ipd.display(ipd.Audio(waveform.squeeze().numpy(), rate=sr))

## Step 5: Encode Audio with Mimi Codec

Encode both source audio and synthesized target audio through the Mimi
neural codec to get discrete token representations.

Output: `.npy` files of shape `(Q=8, T)` -- 8 codebooks x T timesteps.

In [ ]:
AUDIO_TOKEN_DIR = f'{BASE_DIR}/audio_tokens'

# Encode source audio
if SOURCE == 'kenspeech':
    print('=== Encoding source audio (KenSpeech Swahili) ===')
    !cd {REPO_DIR} && python data/prepare/encode_audio.py \
        --source kenspeech \
        --kenspeech_dir {KENSPEECH_DIR} \
        --output_dir {AUDIO_TOKEN_DIR}/kenspeech_sw \
        --num_codebooks 8 \
        --max_samples {MAX_SAMPLES} \
        --max_duration 20.0
else:
    print('=== Encoding source audio (FLEURS English) ===')
    !cd {REPO_DIR} && python data/prepare/encode_audio.py \
        --source directory \
        --audio_dir {FLEURS_AUDIO_DIR} \
        --output_dir {AUDIO_TOKEN_DIR}/fleurs_en \
        --num_codebooks 8 \
        --max_duration 20.0

In [ ]:
# Encode synthesized target audio
synth_wav_dir = f'{SYNTH_DIR}/aligned_audio'
if not os.path.exists(synth_wav_dir):
    synth_wav_dir = f'{SYNTH_DIR}/raw_synthesis/wavs'

print(f'=== Encoding synthesized {TARGET_LANG} audio ===')
!cd {REPO_DIR} && python data/prepare/encode_audio.py \
    --source directory \
    --audio_dir {synth_wav_dir} \
    --output_dir {AUDIO_TOKEN_DIR}/synth_{TARGET_LANG} \
    --num_codebooks 8 \
    --max_duration 30.0

In [ ]:
# Verify encoded tokens
import numpy as np

source_subdir = 'kenspeech_sw' if SOURCE == 'kenspeech' else f'fleurs_{SOURCE_LANG}'
for subdir in [source_subdir, f'synth_{TARGET_LANG}']:
    token_dir = f'{AUDIO_TOKEN_DIR}/{subdir}'
    if os.path.exists(token_dir):
        npy_files = list(Path(token_dir).glob('*.npy'))
        print(f'\n{subdir}: {len(npy_files)} token files')
        if npy_files:
            sample = np.load(str(npy_files[0]))
            print(f'  Sample shape: {sample.shape} (Q={sample.shape[0]}, T={sample.shape[1]})')
            print(f'  Duration: ~{sample.shape[1] / 12.5:.1f}s at 12.5Hz')

## Step 6: Pipeline Summary

Check all outputs and prepare for training.

In [ ]:
# Push pipeline output to HuggingFace Hub for persistence
# Uploads BASE_DIR (/kaggle/working/hibiki-sw) to your HF dataset repo
# Set HF_REPO to your target repo, e.g. "victormugambi/hibiki-sw-pipeline-output"
#
# IMPORTANT: HF_TOKEN must have WRITE access.
# In HuggingFace Settings > Access Tokens, create a token with "write" role.
# Then add it to Kaggle: Settings > Secrets > HF_TOKEN
from huggingface_hub import HfApi
import os

HF_REPO = "victormugambi/hibiki-sw-pipeline-output"  # change if needed
DIRECTION = f'{SOURCE_LANG}2{TARGET_LANG}'

hf_token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")
if not hf_token:
    print("WARNING: HF_TOKEN not set. Cannot push to HuggingFace.")
    print("Add your write-access token as Kaggle secret 'HF_TOKEN'.")
    print("Skipping push. Save the output manually via 'Save Version' in Kaggle.")
else:
    api = HfApi(token=hf_token)

    # Create the repo if it doesn't exist (403 means token lacks write access)
    try:
        api.create_repo(repo_id=HF_REPO, repo_type="dataset", exist_ok=True, private=True)
        print(f"Repo ready: hf://datasets/{HF_REPO}")
    except Exception as e:
        if "403" in str(e) or "Forbidden" in str(e):
            print(f"ERROR: 403 Forbidden when creating repo '{HF_REPO}'.")
            print("Your HF_TOKEN does not have write/create permissions.")
            print("Fix: Go to huggingface.co > Settings > Access Tokens > create a 'write' token.")
            print("Then update the HF_TOKEN Kaggle secret and rerun.")
            print("\nFalling back: saving a summary of outputs instead.")
        else:
            print(f"ERROR creating repo: {e}")
        # Don't raise — show what was produced instead of crashing
        hf_token = None  # signal to skip upload below

    if hf_token:
        print(f"Uploading {BASE_DIR} -> hf://datasets/{HF_REPO}/{DIRECTION}/")
        print("This may take several minutes depending on output size...")

        try:
            api.upload_folder(
                folder_path=BASE_DIR,
                repo_id=HF_REPO,
                repo_type="dataset",
                path_in_repo=DIRECTION,          # e.g. sw2en/ or en2sw/
                commit_message=f"Pipeline output: {DIRECTION}",
                ignore_patterns=["*.pyc", "__pycache__"],
            )
            print(f"\nDone! Output saved to:")
            print(f"  https://huggingface.co/datasets/{HF_REPO}/tree/main/{DIRECTION}")
        except Exception as e:
            print(f"ERROR uploading: {e}")
            print("Data is still in /kaggle/working — use 'Save Version' to persist it as a Kaggle dataset.")

# Summarize what was produced regardless of push outcome
print("\n=== Output Summary ===")
from pathlib import Path
for name, path in [
    ("Transcriptions", f"{BASE_DIR}/transcriptions/{SOURCE_LANG}"),
    ("Translations",   f"{BASE_DIR}/translations/{DIRECTION}"),
    ("Synthesized audio", f"{BASE_DIR}/synthetic_audio/{DIRECTION}"),
    ("Source tokens",  f"{BASE_DIR}/audio_tokens/{'kenspeech_sw' if SOURCE == 'kenspeech' else 'fleurs_en'}"),
    (f"Synth tokens",  f"{BASE_DIR}/audio_tokens/synth_{TARGET_LANG}"),
]:
    if os.path.exists(path):
        n_json = len(list(Path(path).rglob('*.json')))
        n_wav  = len(list(Path(path).rglob('*.wav')))
        n_npy  = len(list(Path(path).rglob('*.npy')))
        parts = [f"{n} {ext}" for n, ext in [(n_json,'json'),(n_wav,'wav'),(n_npy,'npy')] if n]
        print(f"  {name}: {', '.join(parts) or '(empty)'}")
    else:
        print(f"  {name}: [NOT FOUND]")

## Step 7: Persist Output

Kaggle wipes `/kaggle/working` when the session ends. Save the pipeline output to HuggingFace Hub **before closing the session**.

In [ ]:
print('=' * 60)
print(f'PIPELINE COMPLETE: {SOURCE_LANG} -> {TARGET_LANG}')
print('=' * 60)

artifacts = {
    'Transcriptions': f'{BASE_DIR}/transcriptions/{SOURCE_LANG}',
    'Translations': f'{BASE_DIR}/translations/{DIRECTION}',
    'Alignments': f'{BASE_DIR}/alignments/{DIRECTION}',
    'Synthesized audio': SYNTH_DIR,
    f'Source tokens': f'{AUDIO_TOKEN_DIR}/{"kenspeech_sw" if SOURCE == "kenspeech" else "fleurs_en"}',
    f'Target tokens (synth_{TARGET_LANG})': f'{AUDIO_TOKEN_DIR}/synth_{TARGET_LANG}',
}

for name, path in artifacts.items():
    if os.path.exists(path):
        n_json = len(list(Path(path).rglob('*.json')))
        n_wav = len(list(Path(path).rglob('*.wav')))
        n_npy = len(list(Path(path).rglob('*.npy')))
        parts = []
        if n_json: parts.append(f'{n_json} json')
        if n_wav: parts.append(f'{n_wav} wav')
        if n_npy: parts.append(f'{n_npy} npy')
        print(f'  {name}: {", ".join(parts)}')
    else:
        print(f'  {name}: [NOT FOUND]')

print(f'\n=== Next Steps ===')
if SOURCE_LANG == 'sw':
    print('1. Re-run this notebook with SOURCE_LANG="en", TARGET_LANG="sw"')
else:
    print('1. Both directions are done!')
print('2. Run notebook 00_data_preparation.ipynb for tokenizer + text tokens')
print('3. Create S2ST manifest with create_s2st_manifest.py')
print('4. Start Stages 1-2 training on Kaggle')